In [ ]:
"""
C2/C3 - Nettoyage et préparation des données FINESS
=====================================================
- Normalisation des colonnes (noms, types)
- Gestion des doublons
- Nettoyage des valeurs nulles et formats
- Filtrage Occitanie (région 76, départements 09,11,12,30,31,32,34,46,48,65,66,81,82)
- Règles d'agrégation pour combiner les 3 sources
- Géoréférencement (Lambert93 → WGS84)
"""
import logging
import os
import plotly.express as px
import pandas as pd
import math
from pyproj import Transformer

from pathlib import Path

#BASE_DIR = Path(__file__).resolve().parent  # dossier du .py
df_soins = pd.read_csv( "../data/raw/finess_activites_soins.csv", sep=";", encoding="utf-8")
df_communes = pd.read_csv("../data/geo/communes-france-2025.csv", sep=",", encoding="utf-8")
df_etab = pd.read_csv("../data/raw/finess_etablissements.txt", sep=";", encoding="utf-8")


df_soins = pd.read_csv(BASE_DIR / "../data/raw/finess_activites_soins.csv", sep=";", encoding="utf-8")
df_communes = pd.read_csv(BASE_DIR / "../data/geo/communes-france-2025.csv", sep=",", encoding="utf-8", 1: "string",  # colonne 0
        9: "string"   # colonne 1
        11: "string"   # colonne 1
        13: "string"   # colonne 1
        17: "string"   # colonne 1
    })
df_etab = pd.read_csv(BASE_DIR / "../data/raw/finess_etablissements.txt", sep=";", encoding="utf-8", dtype={
        0: "string",  # colonne 0
        1: "string"   # colonne 1
        12: "string"   # colonne 1
    })
df_equi = pd.read_csv(BASE_DIR / "../data/raw/finess_equipements_sociaux.csv", sep=";", encoding="utf-8", dtype={
        0: "string",  # colonne 0
        1: "string"   # colonne 1
    })


# ---------------------------------------------------------------------------
# Fonctions de nettoyage
# ---------------------------------------------------------------------------

def clean_nofiness(series: pd.Series) -> pd.Series:
    """
    Normalise les numéros FINESS : chaîne de 9 caractères, zéros à gauche.
    Exemples : 10001246 → '010001246', 310001234 → '310001234'
    """
    return series.astype(str).str.zfill(9)


def clean_department(code: str) -> str:
    """Normalise un code département : '9' → '09', '31' → '31'."""
    if pd.isna(code):
        return None
    code = str(code).strip().zfill(2)
    return code[:2]  # Garde uniquement les 2 premiers chiffres


def lambert93_to_wgs84(x: float, y: float):
    """
    Convertit des coordonnées Lambert93 (EPSG:2154) en WGS84 (lat/lon).
    Implémentation pure Python basée sur les paramètres IAG GRS80.
    """
    try:
        if pd.isna(x) or pd.isna(y) or float(x) == 0 or float(y) == 0:
            return None, None
        x, y = float(x), float(y)

        # Paramètres ellipsoïde GRS80
        a = 6378137.0
        e = 0.0818191910428158

        # Paramètres projection Lambert93
        lc = math.radians(3.0)          # Longitude centrale
        phi0 = math.radians(46.5)       # Latitude d'origine
        phi1 = math.radians(44.0)       # Parallèle 1
        phi2 = math.radians(49.0)       # Parallèle 2
        x0 = 700000.0
        y0 = 6600000.0

        def _geo_lat(phi):
            sp = e * math.sin(phi)
            return math.tan(math.pi / 4 + phi / 2) * ((1 - sp) / (1 + sp)) ** (e / 2)

        m1 = math.cos(phi1) / math.sqrt(1 - (e * math.sin(phi1)) ** 2)
        m2 = math.cos(phi2) / math.sqrt(1 - (e * math.sin(phi2)) ** 2)
        t1 = _geo_lat(phi1)
        t2 = _geo_lat(phi2)
        t0 = _geo_lat(phi0)

        n = math.log(m1 / m2) / math.log(t1 / t2)
        F = m1 / (n * t1 ** n)
        r0 = a * F * t0 ** n

        dx, dy = x - x0, y - y0 + r0
        r = math.sqrt(dx ** 2 + dy ** 2) * math.copysign(1, n)
        theta = math.atan(dx / (r0 - (y - y0)))
        lon = math.degrees(theta / n + lc)

        t = (r / (a * F)) ** (1 / n)
        phi = math.pi / 2 - 2 * math.atan(t)
        for _ in range(10):
            sp = e * math.sin(phi)
            phi = math.pi / 2 - 2 * math.atan(t * ((1 - sp) / (1 + sp)) ** (e / 2))

        lat = math.degrees(phi)

        if 41 <= lat <= 52 and -5 <= lon <= 10:
            return round(lat, 6), round(lon, 6)
        return None, None
    except Exception:
        return None, None

   



df_soins.head()




/tmp/ipykernel_3050/1562403606.py:22: DtypeWarning: Columns (1,9,11,13,17) have mixed types. Specify dtype option on import or set low_memory=False.
  df_communes = pd.read_csv("../data/geo/communes-france-2025.csv", sep=",", encoding="utf-8")
/tmp/ipykernel_3050/1562403606.py:23: DtypeWarning: Columns (0,1,12) have mixed types. Specify dtype option on import or set low_memory=False.
  df_etab = pd.read_csv("../data/raw/finess_etablissements.txt", sep=";", encoding="utf-8")


,nofinessej,rsej,activite,libactivite,modalite,libmodalite,forme,libforme,noautor,dateautor,nofinesset,rset,noligautor,noligautoranc,noautorarhgos,noimplarhgos,sectpsy,libsectpsy,datemeo,datefin
0,010780054,CH FLEYRIAT,1,Médecine,00,Pas de modalité,1,Hospitalisation complète (24 heures consécutiv...,842601862,2000-06-07,010000024,CH DE FLEYRIAT,1862,NaN,84-82-43,84-82-43,NaN,NaN,2001-08-03,2029-02-01
1,010780054,CH FLEYRIAT,1,Médecine,00,Pas de modalité,2,Hospitalisation à temps partiel de jour ou de ...,842601863,2002-03-13,010000024,CH DE FLEYRIAT,1863,NaN,84-82-42,84-82-42,NaN,NaN,2006-04-21,2029-02-01
2,010780054,CH FLEYRIAT,3,"Gynécologie, obstétrique, néonatologie, réanim...",01,Gynécologie obstétrique,1,Hospitalisation complète (24 heures consécutiv...,842604656,2001-01-10,010000024,CH DE FLEYRIAT,4656,NaN,84-82-46,84-82-46,NaN,NaN,2001-08-03,2029-02-01
3,010780054,CH FLEYRIAT,3,"Gynécologie, obstétrique, néonatologie, réanim...",02,Néonatologie sans soins intensifs,1,Hospitalisation complète (24 heures consécutiv...,842604657,2001-01-10,010000024,CH DE FLEYRIAT,4657,NaN,84-82-47,84-82-47,NaN,NaN,2006-04-21,2029-02-01
4,010780054,CH FLEYRIAT,3,"Gynécologie, obstétrique, néonatologie, réanim...",03,Néonatologie avec soins intensifs,1,Hospitalisation complète (24 heures consécutiv...,842604658,2001-01-10,010000024,CH DE FLEYRIAT,4658,NaN,84-82-48,84-82-48,NaN,NaN,2006-04-21,2029-02-01


In [37]:
#nettoyage activité soins
# Sélection des colonnes utiles
cols = ["nofinesset", "rsej", "libactivite", "libmodalite", "libforme", "datefin"]
df_soins = df_soins[[c for c in cols if c in df_soins.columns]].copy()
#nettoyage nofinesset
df_soins["nofinesset"] = clean_nofiness(df_soins["nofinesset"])


In [38]:
# Regroupement des catégories d'activités pour les soins
def regrouper_categorie(cat):
    cat = cat.lower()

    if any(x in cat for x in ["greffe"]):
        return "Greffe"

    if any(x in cat for x in ["soins de suite"]):
        return "Soins de suite"

    if any(x in cat for x in ["chirurgie"]):
        return "Chirurgie"
    if any(x in cat for x in ["médecine d\'urgence"]):
        return "Médecine d'urgence"
    if any(x in cat for x in ["médecine"]):
        return "Médecine"
    if any(x in cat for x in ["médecine"]):
        return "Médecine"
    if any(x in cat for x in ["psychiatrie"]):
        return "Psychiatrie"
    if any(x in cat for x in ["amp dpn"]):
        return "AMP DPN"
    if any(x in cat for x in ["cancer"]):
        return "Cancer"
    if any(x in cat for x in ["gynécologie"]):
        return "Gynécologie"
    if any(x in cat for x in ["soins de longue durée"]):
        return "Soins de longue durée"
    if any(x in cat for x in ["insuffisance rénale"]):
        return "Insuffisance rénale"
    if any(x in cat for x in ["examen des caractéristiques génétiques"]):
        return "Examen des caractéristiques génétiques"
    return "Autres"

# Application
df_soins["groupe_activites"] = df_soins["libactivite"].apply(regrouper_categorie)

#enlever les doublons 
df_soins = df_soins.drop_duplicates()

In [39]:
df_etab.head()

,nofinesset,nofinessej,rs,rslongue,complrs,compldistrib,numvoie,typvoie,voie,compvoie,...,codesph,libsph,dateouv,dateautor,datemaj,numuai,coordxet,coordyet,sourcecoordet,datemaj_geoloc
0,10000024,10780054,CH DE FLEYRIAT,CENTRE HOSPITALIER DE BOURG-EN-BRESSE FLEYRIAT,NaN,NaN,900.0,RTE,DE PARIS,NaN,...,1.0,Etablissement public de santé,1979-02-13,1979-02-13,2020-02-04,NaN,870262.2,6571540.8,"1,ATLASANTE,96,BAN,EPSG:2154 RGF93 / Lambert-9...",2026-01-05
1,10000032,10780062,CH BUGEY SUD,CENTRE HOSPITALIER BUGEY SUD,NaN,NaN,700.0,AV,DE NARVIK,NaN,...,1.0,Etablissement public de santé,1901-01-01,1901-01-01,2021-07-07,NaN,908313.3,6520061.2,"1,ATLASANTE,95,BAN,EPSG:2154 RGF93 / Lambert-9...",2026-01-05
2,10000065,10780096,CH DE TREVOUX - MONTPENSIER,CENTRE HOSPITALIER DE TREVOUX - MONTPENSIER,NaN,NaN,14.0,R,DE L'HOPITAL,NaN,...,1.0,Etablissement public de santé,1901-01-01,1901-01-01,2018-01-12,NaN,837272.3,6539470.4,"1,ATLASANTE,85,BAN,EPSG:2154 RGF93 / Lambert-9...",2026-01-05
3,10000081,10780112,CH DU PAYS DE GEX,CENTRE HOSPITALIER DU PAYS DE GEX,NaN,NaN,160.0,R,MARC PANISSOD,NaN,...,1.0,Etablissement public de santé,1901-01-01,1901-01-01,2020-02-04,NaN,935221.3,6584833.3,"1,ATLASANTE,85,BAN,EPSG:2154 RGF93 / Lambert-9...",2026-01-05
4,10000099,10780120,CH DE MEXIMIEUX,CENTRE HOSPITALIER DE MEXIMIEUX,NaN,NaN,13.0,AV,DU DOCTEUR BOYER,NaN,...,1.0,Etablissement public de santé,1945-01-01,1945-01-01,2020-06-30,NaN,870118.2,6536469.4,"1,ATLASANTE,96,BAN,EPSG:2154 RGF93 / Lambert-9...",2026-01-05


In [40]:
#verification des codes communes : 3 chiffres
df_etab["commune"] = (
    df_etab["commune"]
        .astype("Int64")  # gère les NA proprement
        .astype(str)
        .str.zfill(3)
)

In [41]:
#nettoyage établissement
# Sélection des colonnes utiles
cols = ["nofinesset", "nofinessej","rs", "rslongue", "departement","commune","complrs","compldistrib","numvoie","typvoie","voie","compvoie","lieuditbp", "libdepartement", "libcategetab", "libcategagretab","coordxet", "coordyet"]
df_etab = df_etab[[c for c in cols if c in df_etab.columns]].copy()
#nettoyage nofinesset
df_etab["nofinesset"] = clean_nofiness(df_etab["nofinesset"])

In [42]:
def clean_department(code: str) -> str:
    """Normalise un code département : '9' → '09', '31' → '31'."""
    if pd.isna(code):
        return None
    code = str(code).strip().zfill(2)
    return code[:2]  # Garde uniquement les 2 premiers chiffres

df_etab["departement"] = df_etab["departement"].apply(clean_department)

In [43]:
df_etab.head()

,nofinesset,nofinessej,rs,rslongue,departement,commune,complrs,compldistrib,numvoie,typvoie,voie,compvoie,lieuditbp,libdepartement,libcategetab,libcategagretab,coordxet,coordyet
0,010000024,10780054,CH DE FLEYRIAT,CENTRE HOSPITALIER DE BOURG-EN-BRESSE FLEYRIAT,01,451,NaN,NaN,900.0,RTE,DE PARIS,NaN,NaN,AIN,Centre Hospitalier (C.H.),Centres Hospitaliers,870262.2,6571540.8
1,010000032,10780062,CH BUGEY SUD,CENTRE HOSPITALIER BUGEY SUD,01,034,NaN,NaN,700.0,AV,DE NARVIK,NaN,BP 139,AIN,Centre Hospitalier (C.H.),Centres Hospitaliers,908313.3,6520061.2
2,010000065,10780096,CH DE TREVOUX - MONTPENSIER,CENTRE HOSPITALIER DE TREVOUX - MONTPENSIER,01,427,NaN,NaN,14.0,R,DE L'HOPITAL,NaN,BP 615,AIN,Centre Hospitalier (C.H.),Centres Hospitaliers,837272.3,6539470.4
3,010000081,10780112,CH DU PAYS DE GEX,CENTRE HOSPITALIER DU PAYS DE GEX,01,173,NaN,NaN,160.0,R,MARC PANISSOD,NaN,BP 437,AIN,"Centre hospitalier, ex Hôpital local",Hôpitaux Locaux,935221.3,6584833.3
4,010000099,10780120,CH DE MEXIMIEUX,CENTRE HOSPITALIER DE MEXIMIEUX,01,244,NaN,NaN,13.0,AV,DU DOCTEUR BOYER,NaN,NaN,AIN,"Centre hospitalier, ex Hôpital local",Hôpitaux Locaux,870118.2,6536469.4


In [44]:
df_equi = pd.read_csv( "../data/raw/finess_equipements_sociaux.csv", sep=";", encoding="utf-8")

/tmp/ipykernel_3050/2998991303.py:1: DtypeWarning: Columns (0,1) have mixed types. Specify dtype option on import or set low_memory=False.
  df_equi = pd.read_csv( "../data/raw/finess_equipements_sociaux.csv", sep=";", encoding="utf-8")


In [45]:
df_equi.head()

,nofinesset,nofinessej,de,libde,ta,libta,client,libclient,sourceinfo,capinstot,...,capautot,capautm,capautf,capauthab,ageminiaut,agemaxiaut,indsupaut,dateautor,datemajaut,datemajinst
0,010001246,010790368,926,Hébergement résidence autonomie personnes âgée...,11,Hébergement Complet Internat,701,Personnes Agées Autonomes,R,4.0,...,4.0,NaN,NaN,NaN,NaN,NaN,N,2017-01-03,2017-04-21,2017-02-08
1,010001246,010790368,927,Hébergement résidence autonomie personnes âgée...,11,Hébergement Complet Internat,701,Personnes Agées Autonomes,9,5.0,...,5.0,NaN,NaN,0.0,NaN,NaN,N,2017-01-03,2017-04-21,2017-02-08
2,010001261,010785897,908,Aide par le travail pour Adultes Handicapés,13,Semi-Internat,110,Déficience Intellectuelle (sans autre indication),9,50.0,...,50.0,NaN,NaN,NaN,NaN,NaN,N,2017-01-03,2017-04-21,2004-06-15
3,010001295,010784270,259,Activ.Club et Équipe de Prévention,16,Prestation en milieu ordinaire,805,Jeunes et familles en risque d'inadaptation so...,9,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,N,2017-01-03,2017-04-21,2004-06-15
4,010001360,010001352,657,Accueil temporaire pour Personnes Âgées,11,Hébergement Complet Internat,701,Personnes Agées Autonomes,R,1.0,...,1.0,NaN,NaN,0.0,NaN,NaN,O,2005-11-07,2017-03-03,2017-03-03


In [46]:
import unicodedata

def clean_text(s):
    if not isinstance(s, str):
        return ""
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode("utf-8")
    s = s.replace("’", "'")
    s = " ".join(s.split())
    return s.lower()

def regrouper_categorie(cat):
    cat = clean_text(cat)

    if any(x in cat for x in [
        "ehpad", "personnes agees", "hebergement pour personnes agees", "maison de retraite", "hbergement pour personnes agees dependantes"
    ]):
        return "Personnes âgées"

    if any(x in cat for x in [
        "ime", "itep", "sessad", "enfance", "enfant", "educatif", "pedagogique", "cmpp", "scolarisation", "educative", "c.m.p.p."
    ]):
        return "Enfance"

    if any(x in cat for x in [
        "handicap", "handicape", "handicapes", "handicapees", "adapte"
    ]):
        return "Handicap"

    if any(x in cat for x in [
        "soins a domicile", "aide a domicile", "service a domicile", "a domicile"
    ]):
        return "Aide et soins à domicile"

    if any(x in cat for x in [
        "psychologique", "psy", "psychiatrie", "cmpp"
    ]):
        return "Aide psychologique"

    if any(x in cat for x in [
        "hebergement", "maison relais", "pension de famille", "maisons relais", "accueil temporaire"
    ]):
        return "Hébergement"

    if any(x in cat for x in [
        "social", "clico", "service social", "intervention educative", "club prevention"
    ]):
        return "Social"

    if any(x in cat for x in [
        "reinsertion", "adaptation", "rehabilitation", "protection des majeurs"
    ]):
        return "Réinsertion et adaptation"
    
    if any(x in cat for x in [
        "aidant", "aidants"
    ]):
        return "Aidants"
    if any(x in cat for x in [
        "cure"
    ]):
        return "Cure"

    return "Autres"


# Application
df_equi["groupe_equipement"] = df_equi["libde"].apply(regrouper_categorie)

In [47]:
df_equi = df_equi.drop_duplicates()

In [48]:
df_equi.head()

,nofinesset,nofinessej,de,libde,ta,libta,client,libclient,sourceinfo,capinstot,...,capautm,capautf,capauthab,ageminiaut,agemaxiaut,indsupaut,dateautor,datemajaut,datemajinst,groupe_equipement
0,010001246,010790368,926,Hébergement résidence autonomie personnes âgée...,11,Hébergement Complet Internat,701,Personnes Agées Autonomes,R,4.0,...,NaN,NaN,NaN,NaN,NaN,N,2017-01-03,2017-04-21,2017-02-08,Personnes âgées
1,010001246,010790368,927,Hébergement résidence autonomie personnes âgée...,11,Hébergement Complet Internat,701,Personnes Agées Autonomes,9,5.0,...,NaN,NaN,0.0,NaN,NaN,N,2017-01-03,2017-04-21,2017-02-08,Personnes âgées
2,010001261,010785897,908,Aide par le travail pour Adultes Handicapés,13,Semi-Internat,110,Déficience Intellectuelle (sans autre indication),9,50.0,...,NaN,NaN,NaN,NaN,NaN,N,2017-01-03,2017-04-21,2004-06-15,Handicap
3,010001295,010784270,259,Activ.Club et Équipe de Prévention,16,Prestation en milieu ordinaire,805,Jeunes et familles en risque d'inadaptation so...,9,NaN,...,NaN,NaN,NaN,NaN,NaN,N,2017-01-03,2017-04-21,2004-06-15,Autres
4,010001360,010001352,657,Accueil temporaire pour Personnes Âgées,11,Hébergement Complet Internat,701,Personnes Agées Autonomes,R,1.0,...,NaN,NaN,0.0,NaN,NaN,O,2005-11-07,2017-03-03,2017-03-03,Personnes âgées


In [49]:
# Sélection des colonnes utiles
cols = ["nofinesset", "nofinessej","libde", "libta", "libclient", "capinstot", "groupe_equipement"]
df_equi = df_equi[[c for c in cols if c in df_equi.columns]].copy()
#nettoyage nofinesset
df_equi["nofinesset"] = clean_nofiness(df_equi["nofinesset"])

In [50]:
df_equi.head()

,nofinesset,nofinessej,libde,libta,libclient,capinstot,groupe_equipement
0,010001246,010790368,Hébergement résidence autonomie personnes âgée...,Hébergement Complet Internat,Personnes Agées Autonomes,4.0,Personnes âgées
1,010001246,010790368,Hébergement résidence autonomie personnes âgée...,Hébergement Complet Internat,Personnes Agées Autonomes,5.0,Personnes âgées
2,010001261,010785897,Aide par le travail pour Adultes Handicapés,Semi-Internat,Déficience Intellectuelle (sans autre indication),50.0,Handicap
3,010001295,010784270,Activ.Club et Équipe de Prévention,Prestation en milieu ordinaire,Jeunes et familles en risque d'inadaptation so...,NaN,Autres
4,010001360,010001352,Accueil temporaire pour Personnes Âgées,Hébergement Complet Internat,Personnes Agées Autonomes,1.0,Personnes âgées


In [51]:
#jointure equipements
# LEFT JOIN - garde tout df1, complète avec df2 si existe
df_equi_tot = df_equi.merge(df_etab, on="nofinesset", how="left")

In [52]:
#jointure soins 
df_soins_tot = df_soins.merge(df_etab, on="nofinesset", how="left")

In [53]:
df_soins_tot.head()

,nofinesset,rsej,libactivite,libmodalite,libforme,datefin,groupe_activites,nofinessej,rs,rslongue,...,numvoie,typvoie,voie,compvoie,lieuditbp,libdepartement,libcategetab,libcategagretab,coordxet,coordyet
0,010000024,CH FLEYRIAT,Médecine,Pas de modalité,Hospitalisation complète (24 heures consécutiv...,2029-02-01,Médecine,10780054,CH DE FLEYRIAT,CENTRE HOSPITALIER DE BOURG-EN-BRESSE FLEYRIAT,...,900.0,RTE,DE PARIS,NaN,NaN,AIN,Centre Hospitalier (C.H.),Centres Hospitaliers,870262.2,6571540.8
1,010000024,CH FLEYRIAT,Médecine,Pas de modalité,Hospitalisation à temps partiel de jour ou de ...,2029-02-01,Médecine,10780054,CH DE FLEYRIAT,CENTRE HOSPITALIER DE BOURG-EN-BRESSE FLEYRIAT,...,900.0,RTE,DE PARIS,NaN,NaN,AIN,Centre Hospitalier (C.H.),Centres Hospitaliers,870262.2,6571540.8
2,010000024,CH FLEYRIAT,"Gynécologie, obstétrique, néonatologie, réanim...",Gynécologie obstétrique,Hospitalisation complète (24 heures consécutiv...,2029-02-01,Gynécologie,10780054,CH DE FLEYRIAT,CENTRE HOSPITALIER DE BOURG-EN-BRESSE FLEYRIAT,...,900.0,RTE,DE PARIS,NaN,NaN,AIN,Centre Hospitalier (C.H.),Centres Hospitaliers,870262.2,6571540.8
3,010000024,CH FLEYRIAT,"Gynécologie, obstétrique, néonatologie, réanim...",Néonatologie sans soins intensifs,Hospitalisation complète (24 heures consécutiv...,2029-02-01,Gynécologie,10780054,CH DE FLEYRIAT,CENTRE HOSPITALIER DE BOURG-EN-BRESSE FLEYRIAT,...,900.0,RTE,DE PARIS,NaN,NaN,AIN,Centre Hospitalier (C.H.),Centres Hospitaliers,870262.2,6571540.8
4,010000024,CH FLEYRIAT,"Gynécologie, obstétrique, néonatologie, réanim...",Néonatologie avec soins intensifs,Hospitalisation complète (24 heures consécutiv...,2029-02-01,Gynécologie,10780054,CH DE FLEYRIAT,CENTRE HOSPITALIER DE BOURG-EN-BRESSE FLEYRIAT,...,900.0,RTE,DE PARIS,NaN,NaN,AIN,Centre Hospitalier (C.H.),Centres Hospitaliers,870262.2,6571540.8


In [54]:
#je crée le code insee pour pouvoir faire la jointure avec les communes 
#je crée un colonne code_insee à partir de la colonne code commune et de la colonne departement
df_soins_tot.loc[:, "code_insee"] = (
    df_soins_tot["departement"].astype(str).str.zfill(2)
    + df_soins_tot["commune"].astype(str).str.zfill(3)
)

df_soins_tot.head()

,nofinesset,rsej,libactivite,libmodalite,libforme,datefin,groupe_activites,nofinessej,rs,rslongue,...,typvoie,voie,compvoie,lieuditbp,libdepartement,libcategetab,libcategagretab,coordxet,coordyet,code_insee
0,010000024,CH FLEYRIAT,Médecine,Pas de modalité,Hospitalisation complète (24 heures consécutiv...,2029-02-01,Médecine,10780054,CH DE FLEYRIAT,CENTRE HOSPITALIER DE BOURG-EN-BRESSE FLEYRIAT,...,RTE,DE PARIS,NaN,NaN,AIN,Centre Hospitalier (C.H.),Centres Hospitaliers,870262.2,6571540.8,01451
1,010000024,CH FLEYRIAT,Médecine,Pas de modalité,Hospitalisation à temps partiel de jour ou de ...,2029-02-01,Médecine,10780054,CH DE FLEYRIAT,CENTRE HOSPITALIER DE BOURG-EN-BRESSE FLEYRIAT,...,RTE,DE PARIS,NaN,NaN,AIN,Centre Hospitalier (C.H.),Centres Hospitaliers,870262.2,6571540.8,01451
2,010000024,CH FLEYRIAT,"Gynécologie, obstétrique, néonatologie, réanim...",Gynécologie obstétrique,Hospitalisation complète (24 heures consécutiv...,2029-02-01,Gynécologie,10780054,CH DE FLEYRIAT,CENTRE HOSPITALIER DE BOURG-EN-BRESSE FLEYRIAT,...,RTE,DE PARIS,NaN,NaN,AIN,Centre Hospitalier (C.H.),Centres Hospitaliers,870262.2,6571540.8,01451
3,010000024,CH FLEYRIAT,"Gynécologie, obstétrique, néonatologie, réanim...",Néonatologie sans soins intensifs,Hospitalisation complète (24 heures consécutiv...,2029-02-01,Gynécologie,10780054,CH DE FLEYRIAT,CENTRE HOSPITALIER DE BOURG-EN-BRESSE FLEYRIAT,...,RTE,DE PARIS,NaN,NaN,AIN,Centre Hospitalier (C.H.),Centres Hospitaliers,870262.2,6571540.8,01451
4,010000024,CH FLEYRIAT,"Gynécologie, obstétrique, néonatologie, réanim...",Néonatologie avec soins intensifs,Hospitalisation complète (24 heures consécutiv...,2029-02-01,Gynécologie,10780054,CH DE FLEYRIAT,CENTRE HOSPITALIER DE BOURG-EN-BRESSE FLEYRIAT,...,RTE,DE PARIS,NaN,NaN,AIN,Centre Hospitalier (C.H.),Centres Hospitaliers,870262.2,6571540.8,01451


In [55]:
#fonction pour mettre les codes communes
def create_code_insee(df, dep_col="departement", com_col="commune", new_col="code_insee"):
    """
    Crée un code INSEE à partir des colonnes département et commune.
    - dep_col : colonne du département (2 chiffres)
    - com_col : colonne de la commune (3 chiffres)
    - new_col : nom de la colonne créée
    """
    
    # Vérification des colonnes
    if dep_col not in df.columns or com_col not in df.columns:
        raise ValueError(f"Colonnes manquantes : {dep_col} ou {com_col}")
    
    df = df.copy()
    
    df[new_col] = (
        df[dep_col].astype(str).str.zfill(2)
        + df[com_col].astype(str).str.zfill(3)
    )
    
    return df

In [56]:
df_equi_tot = create_code_insee(df_equi_tot, "departement", "commune")
df_equi_tot.head()

,nofinesset,nofinessej_x,libde,libta,libclient,capinstot,groupe_equipement,nofinessej_y,rs,rslongue,...,typvoie,voie,compvoie,lieuditbp,libdepartement,libcategetab,libcategagretab,coordxet,coordyet,code_insee
0,010001246,010790368,Hébergement résidence autonomie personnes âgée...,Hébergement Complet Internat,Personnes Agées Autonomes,4.0,Personnes âgées,10790368,MARPA DE BAGE-LA-VILLE,MARPA ANNEXE DE LA MARPA DE MANZIAT,...,R,DE LA MAIRIE,NaN,NaN,AIN,Résidences autonomie,Etablissements d'Hébergement pour Personnes Âgées,849751.4,6581477.7,01025
1,010001246,010790368,Hébergement résidence autonomie personnes âgée...,Hébergement Complet Internat,Personnes Agées Autonomes,5.0,Personnes âgées,10790368,MARPA DE BAGE-LA-VILLE,MARPA ANNEXE DE LA MARPA DE MANZIAT,...,R,DE LA MAIRIE,NaN,NaN,AIN,Résidences autonomie,Etablissements d'Hébergement pour Personnes Âgées,849751.4,6581477.7,01025
2,010001261,010785897,Aide par le travail pour Adultes Handicapés,Semi-Internat,Déficience Intellectuelle (sans autre indication),50.0,Handicap,10785897,ESAT LES BROSSES,ESAT LES BROSSES,...,NaN,NaN,NaN,LES BROSSES,AIN,Etablissement et Service d'Aide par le Travail...,Etab.et Services de Travail Protégé pour Adult...,861843.3,6600001.5,01433
3,010001295,010784270,Activ.Club et Équipe de Prévention,Prestation en milieu ordinaire,Jeunes et familles en risque d'inadaptation so...,NaN,Autres,10784270,PREVENTION SPECIALISEE - ADSEA,SERVICE DE PREVENTION SPECIALISEE,...,R,PAUL VERLAINE,NaN,NaN,AIN,Club Equipe de Prévention,Services Concourant à la Protection de l'Enfance,871756.1,6567453.0,01289
4,010001360,010001352,Accueil temporaire pour Personnes Âgées,Hébergement Complet Internat,Personnes Agées Autonomes,1.0,Personnes âgées,10001352,MARPA LA VALETTE,MARPA LA VALETTE,...,RTE,DE MONTCET,NaN,NaN,AIN,Résidences autonomie,Etablissements d'Hébergement pour Personnes Âgées,863648.5,6568876.7,01264


In [57]:

df_equi_tot = create_code_insee(df_equi_tot, "departement", "commune")
df_soins_tot = create_code_insee(df_soins_tot, "departement", "commune")
df_etab = create_code_insee(df_etab, "departement", "commune")

In [58]:
df_equi_tot.head()

,nofinesset,nofinessej_x,libde,libta,libclient,capinstot,groupe_equipement,nofinessej_y,rs,rslongue,...,typvoie,voie,compvoie,lieuditbp,libdepartement,libcategetab,libcategagretab,coordxet,coordyet,code_insee
0,010001246,010790368,Hébergement résidence autonomie personnes âgée...,Hébergement Complet Internat,Personnes Agées Autonomes,4.0,Personnes âgées,10790368,MARPA DE BAGE-LA-VILLE,MARPA ANNEXE DE LA MARPA DE MANZIAT,...,R,DE LA MAIRIE,NaN,NaN,AIN,Résidences autonomie,Etablissements d'Hébergement pour Personnes Âgées,849751.4,6581477.7,01025
1,010001246,010790368,Hébergement résidence autonomie personnes âgée...,Hébergement Complet Internat,Personnes Agées Autonomes,5.0,Personnes âgées,10790368,MARPA DE BAGE-LA-VILLE,MARPA ANNEXE DE LA MARPA DE MANZIAT,...,R,DE LA MAIRIE,NaN,NaN,AIN,Résidences autonomie,Etablissements d'Hébergement pour Personnes Âgées,849751.4,6581477.7,01025
2,010001261,010785897,Aide par le travail pour Adultes Handicapés,Semi-Internat,Déficience Intellectuelle (sans autre indication),50.0,Handicap,10785897,ESAT LES BROSSES,ESAT LES BROSSES,...,NaN,NaN,NaN,LES BROSSES,AIN,Etablissement et Service d'Aide par le Travail...,Etab.et Services de Travail Protégé pour Adult...,861843.3,6600001.5,01433
3,010001295,010784270,Activ.Club et Équipe de Prévention,Prestation en milieu ordinaire,Jeunes et familles en risque d'inadaptation so...,NaN,Autres,10784270,PREVENTION SPECIALISEE - ADSEA,SERVICE DE PREVENTION SPECIALISEE,...,R,PAUL VERLAINE,NaN,NaN,AIN,Club Equipe de Prévention,Services Concourant à la Protection de l'Enfance,871756.1,6567453.0,01289
4,010001360,010001352,Accueil temporaire pour Personnes Âgées,Hébergement Complet Internat,Personnes Agées Autonomes,1.0,Personnes âgées,10001352,MARPA LA VALETTE,MARPA LA VALETTE,...,RTE,DE MONTCET,NaN,NaN,AIN,Résidences autonomie,Etablissements d'Hébergement pour Personnes Âgées,863648.5,6568876.7,01264


In [59]:
from pyproj import Transformer

def convert_lambert_to_wgs84(
    df,
    x_col="coordxet",
    y_col="coordyet",
    lon_col="longitude",
    lat_col="latitude"
):
    """
    Convertit des coordonnées Lambert 93 (EPSG:2154) en WGS84 (EPSG:4326).
    - x_col : colonne des X Lambert 93
    - y_col : colonne des Y Lambert 93
    - lon_col : nom de la colonne longitude créée
    - lat_col : nom de la colonne latitude créée
    """

    # Vérification des colonnes
    if x_col not in df.columns or y_col not in df.columns:
        raise ValueError(f"Colonnes manquantes : {x_col} ou {y_col}")

    df = df.copy()

    # Création du transformateur
    transformer = Transformer.from_crs("EPSG:2154", "EPSG:4326", always_xy=True)

    # Conversion
    df[lon_col], df[lat_col] = transformer.transform(
        df[x_col].astype(float).values,
        df[y_col].astype(float).values
    )

    return df

In [60]:
df_soins_tot = convert_lambert_to_wgs84(df_soins_tot, "coordxet", "coordyet")
df_soins_tot.head()

,nofinesset,rsej,libactivite,libmodalite,libforme,datefin,groupe_activites,nofinessej,rs,rslongue,...,compvoie,lieuditbp,libdepartement,libcategetab,libcategagretab,coordxet,coordyet,code_insee,longitude,latitude
0,010000024,CH FLEYRIAT,Médecine,Pas de modalité,Hospitalisation complète (24 heures consécutiv...,2029-02-01,Médecine,10780054,CH DE FLEYRIAT,CENTRE HOSPITALIER DE BOURG-EN-BRESSE FLEYRIAT,...,NaN,NaN,AIN,Centre Hospitalier (C.H.),Centres Hospitaliers,870262.2,6571540.8,01451,5.209181,46.222286
1,010000024,CH FLEYRIAT,Médecine,Pas de modalité,Hospitalisation à temps partiel de jour ou de ...,2029-02-01,Médecine,10780054,CH DE FLEYRIAT,CENTRE HOSPITALIER DE BOURG-EN-BRESSE FLEYRIAT,...,NaN,NaN,AIN,Centre Hospitalier (C.H.),Centres Hospitaliers,870262.2,6571540.8,01451,5.209181,46.222286
2,010000024,CH FLEYRIAT,"Gynécologie, obstétrique, néonatologie, réanim...",Gynécologie obstétrique,Hospitalisation complète (24 heures consécutiv...,2029-02-01,Gynécologie,10780054,CH DE FLEYRIAT,CENTRE HOSPITALIER DE BOURG-EN-BRESSE FLEYRIAT,...,NaN,NaN,AIN,Centre Hospitalier (C.H.),Centres Hospitaliers,870262.2,6571540.8,01451,5.209181,46.222286
3,010000024,CH FLEYRIAT,"Gynécologie, obstétrique, néonatologie, réanim...",Néonatologie sans soins intensifs,Hospitalisation complète (24 heures consécutiv...,2029-02-01,Gynécologie,10780054,CH DE FLEYRIAT,CENTRE HOSPITALIER DE BOURG-EN-BRESSE FLEYRIAT,...,NaN,NaN,AIN,Centre Hospitalier (C.H.),Centres Hospitaliers,870262.2,6571540.8,01451,5.209181,46.222286
4,010000024,CH FLEYRIAT,"Gynécologie, obstétrique, néonatologie, réanim...",Néonatologie avec soins intensifs,Hospitalisation complète (24 heures consécutiv...,2029-02-01,Gynécologie,10780054,CH DE FLEYRIAT,CENTRE HOSPITALIER DE BOURG-EN-BRESSE FLEYRIAT,...,NaN,NaN,AIN,Centre Hospitalier (C.H.),Centres Hospitaliers,870262.2,6571540.8,01451,5.209181,46.222286


In [61]:

df_equi_tot = convert_lambert_to_wgs84(df_equi_tot, "coordxet", "coordyet")
df_etab = convert_lambert_to_wgs84(df_etab, "coordxet", "coordyet")

In [62]:
df_communes.head()

,Unnamed: 0,code_insee,nom_standard,nom_sans_accent,nom_standard_majuscule,typecom,typecom_texte,reg_code,reg_nom,dep_code,...,code_unite_urbaine,taille_unite_urbaine,population,superficie_hectare,superficie_km2,densite,latitude_centre,longitude_centre,grille_densite,grille_densite_texte
0,0,1001,L'Abergement-Clémenciat,l-abergement-clemenciat,L'ABERGEMENT-CLÉMENCIAT,COM,commune,84,Auvergne-Rhône-Alpes,1,...,1000,0.0,832,1565,16,53.0,46.153,4.926,6,Rural à habitat dispersé
1,1,1002,L'Abergement-de-Varey,l-abergement-de-varey,L'ABERGEMENT-DE-VAREY,COM,commune,84,Auvergne-Rhône-Alpes,1,...,1000,0.0,267,912,9,29.0,46.009,5.428,6,Rural à habitat dispersé
2,2,1004,Ambérieu-en-Bugey,amberieu-en-bugey,AMBÉRIEU-EN-BUGEY,COM,commune,84,Auvergne-Rhône-Alpes,1,...,1303,3.0,14854,2448,24,607.0,45.961,5.373,2,Centres urbains intermédiaires
3,3,1005,Ambérieux-en-Dombes,amberieux-en-dombes,AMBÉRIEUX-EN-DOMBES,COM,commune,84,Auvergne-Rhône-Alpes,1,...,1000,0.0,1897,1605,16,118.0,45.996,4.912,5,Bourgs ruraux
4,4,1006,Ambléon,ambleon,AMBLÉON,COM,commune,84,Auvergne-Rhône-Alpes,1,...,1000,0.0,113,602,6,19.0,45.750,5.594,6,Rural à habitat dispersé


In [63]:
#verification des codes communes : 5 chiffres
df_communes["code_insee"] = (
    df_communes["code_insee"]
        .astype(str)
        .str.zfill(5)
)


In [64]:
#jointures avec communes
#jointures avec communes
df_soins_tot = df_soins_tot.merge(df_communes, on="code_insee", how="left")
df_equi_tot = df_equi_tot.merge(df_communes, on="code_insee", how="left")
df_etab = df_etab.merge(df_communes, on="code_insee", how="left")
df_soins_tot.head()

,nofinesset,rsej,libactivite,libmodalite,libforme,datefin,groupe_activites,nofinessej,rs,rslongue,...,code_unite_urbaine,taille_unite_urbaine,population,superficie_hectare,superficie_km2,densite,latitude_centre,longitude_centre,grille_densite,grille_densite_texte
0,010000024,CH FLEYRIAT,Médecine,Pas de modalité,Hospitalisation complète (24 heures consécutiv...,2029-02-01,Médecine,10780054,CH DE FLEYRIAT,CENTRE HOSPITALIER DE BOURG-EN-BRESSE FLEYRIAT,...,1501,5.0,6808.0,4526.0,45.0,150.0,46.251,5.223,5.0,Bourgs ruraux
1,010000024,CH FLEYRIAT,Médecine,Pas de modalité,Hospitalisation à temps partiel de jour ou de ...,2029-02-01,Médecine,10780054,CH DE FLEYRIAT,CENTRE HOSPITALIER DE BOURG-EN-BRESSE FLEYRIAT,...,1501,5.0,6808.0,4526.0,45.0,150.0,46.251,5.223,5.0,Bourgs ruraux
2,010000024,CH FLEYRIAT,"Gynécologie, obstétrique, néonatologie, réanim...",Gynécologie obstétrique,Hospitalisation complète (24 heures consécutiv...,2029-02-01,Gynécologie,10780054,CH DE FLEYRIAT,CENTRE HOSPITALIER DE BOURG-EN-BRESSE FLEYRIAT,...,1501,5.0,6808.0,4526.0,45.0,150.0,46.251,5.223,5.0,Bourgs ruraux
3,010000024,CH FLEYRIAT,"Gynécologie, obstétrique, néonatologie, réanim...",Néonatologie sans soins intensifs,Hospitalisation complète (24 heures consécutiv...,2029-02-01,Gynécologie,10780054,CH DE FLEYRIAT,CENTRE HOSPITALIER DE BOURG-EN-BRESSE FLEYRIAT,...,1501,5.0,6808.0,4526.0,45.0,150.0,46.251,5.223,5.0,Bourgs ruraux
4,010000024,CH FLEYRIAT,"Gynécologie, obstétrique, néonatologie, réanim...",Néonatologie avec soins intensifs,Hospitalisation complète (24 heures consécutiv...,2029-02-01,Gynécologie,10780054,CH DE FLEYRIAT,CENTRE HOSPITALIER DE BOURG-EN-BRESSE FLEYRIAT,...,1501,5.0,6808.0,4526.0,45.0,150.0,46.251,5.223,5.0,Bourgs ruraux


In [65]:
df_communes.head()

,Unnamed: 0,code_insee,nom_standard,nom_sans_accent,nom_standard_majuscule,typecom,typecom_texte,reg_code,reg_nom,dep_code,...,code_unite_urbaine,taille_unite_urbaine,population,superficie_hectare,superficie_km2,densite,latitude_centre,longitude_centre,grille_densite,grille_densite_texte
0,0,01001,L'Abergement-Clémenciat,l-abergement-clemenciat,L'ABERGEMENT-CLÉMENCIAT,COM,commune,84,Auvergne-Rhône-Alpes,1,...,1000,0.0,832,1565,16,53.0,46.153,4.926,6,Rural à habitat dispersé
1,1,01002,L'Abergement-de-Varey,l-abergement-de-varey,L'ABERGEMENT-DE-VAREY,COM,commune,84,Auvergne-Rhône-Alpes,1,...,1000,0.0,267,912,9,29.0,46.009,5.428,6,Rural à habitat dispersé
2,2,01004,Ambérieu-en-Bugey,amberieu-en-bugey,AMBÉRIEU-EN-BUGEY,COM,commune,84,Auvergne-Rhône-Alpes,1,...,1303,3.0,14854,2448,24,607.0,45.961,5.373,2,Centres urbains intermédiaires
3,3,01005,Ambérieux-en-Dombes,amberieux-en-dombes,AMBÉRIEUX-EN-DOMBES,COM,commune,84,Auvergne-Rhône-Alpes,1,...,1000,0.0,1897,1605,16,118.0,45.996,4.912,5,Bourgs ruraux
4,4,01006,Ambléon,ambleon,AMBLÉON,COM,commune,84,Auvergne-Rhône-Alpes,1,...,1000,0.0,113,602,6,19.0,45.750,5.594,6,Rural à habitat dispersé


In [66]:
def filter_occitanie(df, dep_col="departement"):
    """
    Filtre un DataFrame sur les départements de la région Occitanie.
    - df : DataFrame à filtrer
    - dep_col : nom de la colonne contenant le code département
    """

    departements_occitanie = [
        "09","11","12","30","31","32","34",
        "46","48","65","66","81","82"
    ]

    if dep_col not in df.columns:
        raise ValueError(f"La colonne '{dep_col}' est absente du DataFrame.")

    df = df.copy()
    df[dep_col] = df[dep_col].astype(str).str.zfill(2)

    return df[df[dep_col].isin(departements_occitanie)]


In [67]:
df_etab_occitanie = filter_occitanie(df_etab)

In [68]:
df_soins_tot.head()

,nofinesset,rsej,libactivite,libmodalite,libforme,datefin,groupe_activites,nofinessej,rs,rslongue,...,code_unite_urbaine,taille_unite_urbaine,population,superficie_hectare,superficie_km2,densite,latitude_centre,longitude_centre,grille_densite,grille_densite_texte
0,010000024,CH FLEYRIAT,Médecine,Pas de modalité,Hospitalisation complète (24 heures consécutiv...,2029-02-01,Médecine,10780054,CH DE FLEYRIAT,CENTRE HOSPITALIER DE BOURG-EN-BRESSE FLEYRIAT,...,1501,5.0,6808.0,4526.0,45.0,150.0,46.251,5.223,5.0,Bourgs ruraux
1,010000024,CH FLEYRIAT,Médecine,Pas de modalité,Hospitalisation à temps partiel de jour ou de ...,2029-02-01,Médecine,10780054,CH DE FLEYRIAT,CENTRE HOSPITALIER DE BOURG-EN-BRESSE FLEYRIAT,...,1501,5.0,6808.0,4526.0,45.0,150.0,46.251,5.223,5.0,Bourgs ruraux
2,010000024,CH FLEYRIAT,"Gynécologie, obstétrique, néonatologie, réanim...",Gynécologie obstétrique,Hospitalisation complète (24 heures consécutiv...,2029-02-01,Gynécologie,10780054,CH DE FLEYRIAT,CENTRE HOSPITALIER DE BOURG-EN-BRESSE FLEYRIAT,...,1501,5.0,6808.0,4526.0,45.0,150.0,46.251,5.223,5.0,Bourgs ruraux
3,010000024,CH FLEYRIAT,"Gynécologie, obstétrique, néonatologie, réanim...",Néonatologie sans soins intensifs,Hospitalisation complète (24 heures consécutiv...,2029-02-01,Gynécologie,10780054,CH DE FLEYRIAT,CENTRE HOSPITALIER DE BOURG-EN-BRESSE FLEYRIAT,...,1501,5.0,6808.0,4526.0,45.0,150.0,46.251,5.223,5.0,Bourgs ruraux
4,010000024,CH FLEYRIAT,"Gynécologie, obstétrique, néonatologie, réanim...",Néonatologie avec soins intensifs,Hospitalisation complète (24 heures consécutiv...,2029-02-01,Gynécologie,10780054,CH DE FLEYRIAT,CENTRE HOSPITALIER DE BOURG-EN-BRESSE FLEYRIAT,...,1501,5.0,6808.0,4526.0,45.0,150.0,46.251,5.223,5.0,Bourgs ruraux


In [69]:
df_etab_occitanie.info()

<class 'pandas.core.frame.DataFrame'>
Index: 10007 entries, 5864 to 84437
Data columns (total 48 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   nofinesset                     10007 non-null  object 
 1   nofinessej                     10007 non-null  object 
 2   rs                             10007 non-null  object 
 3   rslongue                       9131 non-null   object 
 4   departement                    10007 non-null  object 
 5   commune                        10007 non-null  object 
 6   complrs                        979 non-null    object 
 7   compldistrib                   1395 non-null   object 
 8   numvoie                        7900 non-null   float64
 9   typvoie                        9364 non-null   object 
 10  voie                           9697 non-null   object 
 11  compvoie                       336 non-null    object 
 12  lieuditbp                      957 non-null    o

In [70]:
df_etab_occ = filter_occitanie(df_etab)
df_soins_occ = filter_occitanie(df_soins_tot)
df_equi_occ = filter_occitanie(df_equi_tot)